# CS-4063 NLP Assignment 3: Transformers + RAG

This notebook implements Parts A, B, and C end-to-end using from-scratch Transformer components (without pretrained Transformer models or banned built-ins).

## Execution Guide
1. Run all cells top to bottom.
2. If running in Google Colab, keep `RUNNING_IN_COLAB = True` and use the provided Google Drive link setup.
3. Outputs are saved to `models/` and `results/`.

## Assignment Coverage
- Part A: Encoder-only Transformer for multitask learning (sentiment + derived feature)
- Part B: Retrieval over training embeddings
- Part C: Decoder-only Transformer for explanation generation with RAG + no-retrieval baseline
- Visualizations: losses, metrics, confusion matrices, retrieval examples, generation comparison

In [ ]:
# Core imports, reproducibility, and runtime setup
import os
import re
import json
import math
import random
import zipfile
from dataclasses import dataclass
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

## Colab Environment Setup

This cell installs helper packages required for downloading and parsing compressed review data in Colab.

In [ ]:
# Install optional dependencies (safe to re-run)
RUNNING_IN_COLAB = 'COLAB_GPU' in os.environ
if RUNNING_IN_COLAB:
    !pip -q install gdown tqdm

## Download and Extract Dataset (Colab-Compatible)

This setup supports your provided Google Drive ZIP link. If files are already present locally, download is skipped.

In [ ]:
# Paths and dataset acquisition
ROOT_DIR = '/content' if RUNNING_IN_COLAB else os.getcwd()
DATA_DIR = os.path.join(ROOT_DIR, 'Dataset')
ZIP_PATH = os.path.join(ROOT_DIR, 'Dataset.zip')
MODELS_DIR = os.path.join(ROOT_DIR, 'models')
RESULTS_DIR = os.path.join(ROOT_DIR, 'results')
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

GDRIVE_FILE_ID = '1in5zybjCKOXXrhev6uowqxGKprzL20wd'

if RUNNING_IN_COLAB and (not os.path.exists(ZIP_PATH)):
    import gdown
    url = f'https://drive.google.com/uc?id={GDRIVE_FILE_ID}'
    gdown.download(url, ZIP_PATH, quiet=False)

if os.path.exists(ZIP_PATH) and (not os.path.exists(DATA_DIR)):
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(ROOT_DIR)

print('ROOT_DIR:', ROOT_DIR)
print('DATA_DIR exists:', os.path.exists(DATA_DIR))

## Build Multi-Category Review Subset

This cell loads metadata from category files, samples approximately 10k-15k reviews per category, and keeps only text + rating.

In [ ]:
# Parse gzipped Amazon review json lines and sample categories
import gzip
from tqdm import tqdm

TARGET_PER_CATEGORY = 12000
MIN_CATEGORIES = 3

candidate_files = []
if os.path.exists(DATA_DIR):
    for f in os.listdir(DATA_DIR):
        if f.startswith('reviews_') and f.endswith('.json.gz'):
            candidate_files.append(os.path.join(DATA_DIR, f))

candidate_files = sorted(candidate_files)
print('Found review files:', len(candidate_files))

selected_files = candidate_files[:MIN_CATEGORIES]
print('Selected categories:', [os.path.basename(x) for x in selected_files])

rows = []
for path in selected_files:
    cat = os.path.basename(path).replace('reviews_', '').replace('_5.json.gz', '')
    loaded = 0
    with gzip.open(path, 'rt', encoding='utf-8') as fh:
        for line in fh:
            try:
                obj = json.loads(line)
                text = (obj.get('reviewText') or '').strip()
                rating = int(obj.get('overall', 0))
                if text and 1 <= rating <= 5:
                    rows.append({'category': cat, 'review_text': text, 'rating': rating})
                    loaded += 1
                if loaded >= TARGET_PER_CATEGORY:
                    break
            except Exception:
                continue

raw_df = pd.DataFrame(rows)
print('Raw subset shape:', raw_df.shape)
raw_df['category'].value_counts()

## Define Targets for Multi-Task Learning

Sentiment target: 1-2 Negative, 3 Neutral, 4-5 Positive. Derived feature target: review length bucket (`short`, `medium`, `long`) computed from token count.

In [ ]:
# Label engineering

def map_sentiment(r):
    if r <= 2:
        return 0  # negative
    if r == 3:
        return 1  # neutral
    return 2      # positive

raw_df = raw_df.drop_duplicates(subset=['review_text']).reset_index(drop=True)
raw_df['sentiment'] = raw_df['rating'].map(map_sentiment)
raw_df['tok_len'] = raw_df['review_text'].str.split().str.len()

q1, q2 = raw_df['tok_len'].quantile([0.33, 0.66]).tolist()
def len_bucket(n):
    if n <= q1:
        return 0
    if n <= q2:
        return 1
    return 2

raw_df['length_bucket'] = raw_df['tok_len'].map(len_bucket)
print(raw_df[['rating', 'sentiment', 'length_bucket']].head())
print('Sentiment distribution:', raw_df['sentiment'].value_counts().to_dict())
print('Length-bucket distribution:', raw_df['length_bucket'].value_counts().to_dict())

## Train/Validation/Test Split (70/15/15)

Data split is stratified by sentiment label to keep class balance.

In [ ]:
# Split dataset
train_df, temp_df = train_test_split(
    raw_df,
    test_size=0.30,
    random_state=SEED,
    stratify=raw_df['sentiment']
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df['sentiment']
)

for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(name, df.shape)

## Text Cleaning, Tokenization, and Vocabulary

Vocabulary is built from training data only, with reserved special tokens.

In [ ]:
# Preprocessing helpers
PAD_TOKEN = '<pad>'
UNK_TOKEN = '<unk>'
BOS_TOKEN = '<bos>'
EOS_TOKEN = '<eos>'
SPECIALS = [PAD_TOKEN, UNK_TOKEN, BOS_TOKEN, EOS_TOKEN]

MAX_LEN = 128
MIN_FREQ = 2

def clean_text(t):
    t = t.lower().strip()
    t = re.sub(r'http\S+|www\.\S+', ' ', t)
    t = re.sub(r'[^a-z0-9\s\.,!?\'\-]', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t

def tokenize(t):
    return t.split()

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

for df in [train_df, val_df, test_df]:
    df['clean_text'] = df['review_text'].map(clean_text)
    df['tokens'] = df['clean_text'].map(tokenize)

counter = Counter()
for toks in train_df['tokens']:
    counter.update(toks)

itos = SPECIALS.copy() + [tok for tok, c in counter.items() if c >= MIN_FREQ]
stoi = {tok: i for i, tok in enumerate(itos)}

PAD_ID = stoi[PAD_TOKEN]
UNK_ID = stoi[UNK_TOKEN]
BOS_ID = stoi[BOS_TOKEN]
EOS_ID = stoi[EOS_TOKEN]

print('Vocab size:', len(itos))

## Numericalization and DataLoaders

Tokens are mapped to indices with truncation/padding to fixed sequence length.

In [ ]:
# Tensor dataset definitions

def encode_tokens(tokens, max_len=MAX_LEN):
    ids = [stoi.get(tok, UNK_ID) for tok in tokens][:max_len]
    attn = [1] * len(ids)
    if len(ids) < max_len:
        pad_count = max_len - len(ids)
        ids += [PAD_ID] * pad_count
        attn += [0] * pad_count
    return ids, attn

class ReviewDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        ids, mask = encode_tokens(row['tokens'])
        return {
            'input_ids': torch.tensor(ids, dtype=torch.long),
            'attention_mask': torch.tensor(mask, dtype=torch.float),
            'sentiment': torch.tensor(int(row['sentiment']), dtype=torch.long),
            'length_bucket': torch.tensor(int(row['length_bucket']), dtype=torch.long),
            'clean_text': row['clean_text']
        }

BATCH_SIZE = 64
train_loader = DataLoader(ReviewDataset(train_df), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(ReviewDataset(val_df), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(ReviewDataset(test_df), batch_size=BATCH_SIZE, shuffle=False)

print('Train batches:', len(train_loader))

## Part A: From-Scratch Transformer Encoder Components

Implements positional encoding, scaled dot-product attention, and custom multi-head self-attention without banned APIs.

In [ ]:
# Positional encoding and attention primitives
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.o_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attention_mask=None):
        bsz, seq_len, _ = x.shape
        q = self.q_proj(x).view(bsz, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(bsz, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(bsz, seq_len, self.n_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(self.head_dim)

        if attention_mask is not None:
            mask = attention_mask.unsqueeze(1).unsqueeze(2)  # [B,1,1,T]
            scores = scores.masked_fill(mask == 0, -1e9)

        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        ctx = torch.matmul(attn, v)

        ctx = ctx.transpose(1, 2).contiguous().view(bsz, seq_len, self.d_model)
        return self.o_proj(ctx)

## Part A: Encoder Block and Multi-Task Model

Shared encoder representation is used for two classification heads and persisted embeddings.

In [ ]:
# Encoder stack and multitask heads
class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadSelfAttention(d_model, n_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attention_mask=None):
        x = self.norm1(x + self.dropout(self.attn(x, attention_mask)))
        x = self.norm2(x + self.dropout(self.ffn(x)))
        return x

class ReviewEncoderModel(nn.Module):
    def __init__(self, vocab_size, d_model=256, n_heads=8, d_ff=512, n_layers=3, dropout=0.1):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.pos_enc = PositionalEncoding(d_model, max_len=MAX_LEN)
        self.blocks = nn.ModuleList([EncoderBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.dropout = nn.Dropout(dropout)

        self.sent_head = nn.Linear(d_model, 3)
        self.feat_head = nn.Linear(d_model, 3)

    def forward(self, input_ids, attention_mask):
        x = self.token_emb(input_ids)
        x = self.pos_enc(x)
        x = self.dropout(x)
        for block in self.blocks:
            x = block(x, attention_mask)

        mask = attention_mask.unsqueeze(-1)
        pooled = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)

        sent_logits = self.sent_head(pooled)
        feat_logits = self.feat_head(pooled)
        return sent_logits, feat_logits, pooled

## Part A: Training and Validation Utilities

Combined loss is used for multi-task optimization.

In [ ]:
# Train/eval loops for Part A
def run_epoch_encoder(model, loader, optimizer=None, alpha=1.0, beta=1.0):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    all_s_true, all_s_pred = [], []
    all_f_true, all_f_pred = [], []

    for batch in loader:
        input_ids = batch['input_ids'].to(DEVICE)
        attn = batch['attention_mask'].to(DEVICE)
        s_y = batch['sentiment'].to(DEVICE)
        f_y = batch['length_bucket'].to(DEVICE)

        with torch.set_grad_enabled(is_train):
            s_logits, f_logits, _ = model(input_ids, attn)
            s_loss = F.cross_entropy(s_logits, s_y)
            f_loss = F.cross_entropy(f_logits, f_y)
            loss = alpha * s_loss + beta * f_loss

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

        total_loss += loss.item()
        all_s_true.extend(s_y.detach().cpu().tolist())
        all_s_pred.extend(s_logits.argmax(-1).detach().cpu().tolist())
        all_f_true.extend(f_y.detach().cpu().tolist())
        all_f_pred.extend(f_logits.argmax(-1).detach().cpu().tolist())

    metrics = {
        'loss': total_loss / max(1, len(loader)),
        'sent_acc': accuracy_score(all_s_true, all_s_pred),
        'feat_acc': accuracy_score(all_f_true, all_f_pred),
        'sent_f1': f1_score(all_s_true, all_s_pred, average='macro'),
        'feat_f1': f1_score(all_f_true, all_f_pred, average='macro')
    }
    return metrics

## Part A: Train Encoder Model

This section trains and stores the best checkpoint based on validation loss.

In [ ]:
# Fit Part A model
encoder_model = ReviewEncoderModel(vocab_size=len(itos)).to(DEVICE)
optimizer = torch.optim.AdamW(encoder_model.parameters(), lr=2e-4, weight_decay=1e-2)

EPOCHS_A = 5
history_a = []
best_val = float('inf')
best_path_a = os.path.join(MODELS_DIR, 'partA_encoder_best.pt')

for epoch in range(1, EPOCHS_A + 1):
    tr = run_epoch_encoder(encoder_model, train_loader, optimizer=optimizer)
    va = run_epoch_encoder(encoder_model, val_loader, optimizer=None)
    history_a.append({'epoch': epoch, **{f'train_{k}': v for k, v in tr.items()}, **{f'val_{k}': v for k, v in va.items()}})
    print(f"Epoch {epoch} | train_loss={tr['loss']:.4f} val_loss={va['loss']:.4f} val_sent_f1={va['sent_f1']:.4f} val_feat_f1={va['feat_f1']:.4f}")

    if va['loss'] < best_val:
        best_val = va['loss']
        torch.save({'model_state_dict': encoder_model.state_dict(), 'vocab': itos}, best_path_a)

print('Saved best encoder to:', best_path_a)

## Part A: Evaluation, Curves, Confusion Matrices, and Embedding Export

Produces test metrics for both tasks and saves training-set embeddings for retrieval.

In [ ]:
# Part A plots and metrics
encoder_model.load_state_dict(torch.load(best_path_a, map_location=DEVICE)['model_state_dict'])
encoder_model.eval()

hist_df = pd.DataFrame(history_a)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist_df['epoch'], hist_df['train_loss'], label='train')
axes[0].plot(hist_df['epoch'], hist_df['val_loss'], label='val')
axes[0].set_title('Part A Loss Curves')
axes[0].legend()
axes[1].plot(hist_df['epoch'], hist_df['train_sent_f1'], label='train_sent_f1')
axes[1].plot(hist_df['epoch'], hist_df['val_sent_f1'], label='val_sent_f1')
axes[1].plot(hist_df['epoch'], hist_df['train_feat_f1'], label='train_feat_f1')
axes[1].plot(hist_df['epoch'], hist_df['val_feat_f1'], label='val_feat_f1')
axes[1].set_title('Part A Macro F1 Curves')
axes[1].legend()
plt.show()

# Test predictions
all_s_true, all_s_pred = [], []
all_f_true, all_f_pred = [], []

with torch.no_grad():
    for batch in test_loader:
        ids = batch['input_ids'].to(DEVICE)
        attn = batch['attention_mask'].to(DEVICE)
        s_y = batch['sentiment'].to(DEVICE)
        f_y = batch['length_bucket'].to(DEVICE)
        s_logits, f_logits, _ = encoder_model(ids, attn)
        all_s_true.extend(s_y.cpu().tolist())
        all_s_pred.extend(s_logits.argmax(-1).cpu().tolist())
        all_f_true.extend(f_y.cpu().tolist())
        all_f_pred.extend(f_logits.argmax(-1).cpu().tolist())

print('Sentiment Report\n', classification_report(all_s_true, all_s_pred, digits=4))
print('Length Bucket Report\n', classification_report(all_f_true, all_f_pred, digits=4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cm_s = confusion_matrix(all_s_true, all_s_pred)
cm_f = confusion_matrix(all_f_true, all_f_pred)
sns.heatmap(cm_s, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Sentiment Confusion Matrix')
sns.heatmap(cm_f, annot=True, fmt='d', cmap='Greens', ax=axes[1])
axes[1].set_title('Length Bucket Confusion Matrix')
plt.show()

# Save train embeddings for retrieval
train_eval_loader = DataLoader(ReviewDataset(train_df), batch_size=BATCH_SIZE, shuffle=False)
embeddings, texts, sent_labels, feat_labels = [], [], [], []
with torch.no_grad():
    for batch in train_eval_loader:
        ids = batch['input_ids'].to(DEVICE)
        attn = batch['attention_mask'].to(DEVICE)
        _, _, pooled = encoder_model(ids, attn)
        embeddings.append(pooled.cpu().numpy())
        texts.extend(batch['clean_text'])
        sent_labels.extend(batch['sentiment'].cpu().tolist())
        feat_labels.extend(batch['length_bucket'].cpu().tolist())

embeddings = np.vstack(embeddings)
np.save(os.path.join(RESULTS_DIR, 'train_embeddings.npy'), embeddings)
pd.DataFrame({'text': texts, 'sentiment': sent_labels, 'length_bucket': feat_labels}).to_csv(
    os.path.join(RESULTS_DIR, 'train_embedding_meta.csv'), index=False
)
hist_df.to_csv(os.path.join(RESULTS_DIR, 'partA_history.csv'), index=False)
print('Saved embeddings and Part A history in results/.')

## Part B: Retrieval Module (Cosine Similarity Top-k)

Builds an in-memory index from saved training embeddings and retrieves semantically similar reviews for each query.

In [ ]:
# Retrieval index and query function
K_RETRIEVE = 3
train_emb = np.load(os.path.join(RESULTS_DIR, 'train_embeddings.npy'))
train_meta = pd.read_csv(os.path.join(RESULTS_DIR, 'train_embedding_meta.csv'))

train_emb_norm = train_emb / (np.linalg.norm(train_emb, axis=1, keepdims=True) + 1e-9)

def encode_text_to_embedding(text):
    toks = tokenize(clean_text(text))
    ids, mask = encode_tokens(toks)
    ids = torch.tensor(ids, dtype=torch.long, device=DEVICE).unsqueeze(0)
    mask = torch.tensor(mask, dtype=torch.float, device=DEVICE).unsqueeze(0)
    with torch.no_grad():
        _, _, pooled = encoder_model(ids, mask)
    q = pooled.squeeze(0).cpu().numpy()
    return q / (np.linalg.norm(q) + 1e-9)

def retrieve_similar_reviews(query_text, k=K_RETRIEVE):
    q = encode_text_to_embedding(query_text)
    sims = np.dot(train_emb_norm, q)
    idx = np.argsort(-sims)[:k]
    return train_meta.iloc[idx].assign(similarity=sims[idx]).reset_index(drop=True)

# Retrieval quality examples
sample_queries = test_df['clean_text'].sample(3, random_state=SEED).tolist()
for i, q in enumerate(sample_queries, 1):
    print(f'\nQuery {i}:', q[:180], '...')
    display(retrieve_similar_reviews(q, k=K_RETRIEVE)[['similarity', 'sentiment', 'length_bucket', 'text']])

## Part C: Construct Explanation Targets and Decoder Inputs

Reference explanations are created with a deterministic template to train the decoder as a language model.

In [ ]:
# Build training text for explanation generation
sent_names = {0: 'negative', 1: 'neutral', 2: 'positive'}
len_names = {0: 'short', 1: 'medium', 2: 'long'}

def build_reference_explanation(sentiment_id, length_id):
    s = sent_names[int(sentiment_id)]
    l = len_names[int(length_id)]
    return f'The review is {s} because the wording and opinions indicate a {s} user experience. The review length is {l}, which supports this interpretation.'

def build_prompt(review_text, sentiment_id, length_id, retrieved_texts=None):
    retrieved_texts = retrieved_texts or []
    ctx = ' || '.join(retrieved_texts)
    return (
        f'[REVIEW] {review_text} '
        f'[PRED_SENT] {sent_names[int(sentiment_id)]} '
        f'[PRED_FEAT] {len_names[int(length_id)]} '
        f'[RETRIEVED] {ctx} '
        f'[EXPLANATION]'
    )

def infer_preds_for_df(df):
    pred_s, pred_f = [], []
    encoder_model.eval()
    loader = DataLoader(ReviewDataset(df), batch_size=BATCH_SIZE, shuffle=False)
    with torch.no_grad():
        for batch in loader:
            ids = batch['input_ids'].to(DEVICE)
            attn = batch['attention_mask'].to(DEVICE)
            s_logits, f_logits, _ = encoder_model(ids, attn)
            pred_s.extend(s_logits.argmax(-1).cpu().tolist())
            pred_f.extend(f_logits.argmax(-1).cpu().tolist())
    out = df.copy().reset_index(drop=True)
    out['pred_sentiment'] = pred_s
    out['pred_length'] = pred_f
    out['reference_expl'] = [build_reference_explanation(s, l) for s, l in zip(pred_s, pred_f)]
    return out

train_gen_df = infer_preds_for_df(train_df)
val_gen_df = infer_preds_for_df(val_df)
test_gen_df = infer_preds_for_df(test_df)

print(train_gen_df[['pred_sentiment', 'pred_length', 'reference_expl']].head(2))

## Part C: Create RAG and Baseline Decoder Datasets

Two training sets are prepared: full RAG (with retrieved context) and baseline (retrieved context removed).

In [ ]:
# Compose LM sequences for decoder training
DEC_MAX_LEN = 180

def make_decoder_records(df, use_retrieval=True, k=K_RETRIEVE):
    records = []
    for _, row in df.iterrows():
        review = row['clean_text']
        if use_retrieval:
            ret = retrieve_similar_reviews(review, k=k)
            ret_texts = ret['text'].tolist()
        else:
            ret_texts = []

        prompt = build_prompt(review, row['pred_sentiment'], row['pred_length'], ret_texts)
        target = row['reference_expl']
        full = f'{BOS_TOKEN} {prompt} {target} {EOS_TOKEN}'
        toks = tokenize(clean_text(full))
        ids = [stoi.get(t, UNK_ID) for t in toks]
        ids = ids[:DEC_MAX_LEN]
        if len(ids) < DEC_MAX_LEN:
            ids += [PAD_ID] * (DEC_MAX_LEN - len(ids))
        records.append(ids)
    return records

train_rag_ids = make_decoder_records(train_gen_df, use_retrieval=True)
val_rag_ids = make_decoder_records(val_gen_df, use_retrieval=True)
test_rag_ids = make_decoder_records(test_gen_df, use_retrieval=True)

train_base_ids = make_decoder_records(train_gen_df, use_retrieval=False)
val_base_ids = make_decoder_records(val_gen_df, use_retrieval=False)
test_base_ids = make_decoder_records(test_gen_df, use_retrieval=False)

print('RAG train sequences:', len(train_rag_ids), '| Baseline train sequences:', len(train_base_ids))

## Part C: Decoder-Only Transformer (From Scratch)

Implements masked self-attention so tokens cannot attend to future positions.

In [ ]:
# Decoder dataset and model components
class DecoderDataset(Dataset):
    def __init__(self, seqs):
        self.seqs = seqs
    def __len__(self):
        return len(self.seqs)
    def __getitem__(self, idx):
        x = torch.tensor(self.seqs[idx], dtype=torch.long)
        return x

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_model = d_model
        self.head_dim = d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.o_proj = nn.Linear(d_model, d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, pad_mask=None):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        scores = (q @ k.transpose(-1, -2)) / math.sqrt(self.head_dim)
        causal = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        scores = scores.masked_fill(causal.unsqueeze(0).unsqueeze(0), -1e9)

        if pad_mask is not None:
            mask = pad_mask.unsqueeze(1).unsqueeze(2)
            scores = scores.masked_fill(mask == 0, -1e9)

        w = torch.softmax(scores, dim=-1)
        w = self.drop(w)
        out = w @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.o_proj(out)

class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn = CausalSelfAttention(d_model, n_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.ReLU(), nn.Dropout(dropout), nn.Linear(d_ff, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, pad_mask=None):
        x = self.norm1(x + self.drop(self.attn(x, pad_mask)))
        x = self.norm2(x + self.drop(self.ffn(x)))
        return x

class ReviewExplanationDecoder(nn.Module):
    def __init__(self, vocab_size, d_model=256, n_heads=8, d_ff=512, n_layers=3, dropout=0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.pos = PositionalEncoding(d_model, max_len=DEC_MAX_LEN)
        self.blocks = nn.ModuleList([DecoderBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, input_ids):
        pad_mask = (input_ids != PAD_ID).float()
        x = self.tok_emb(input_ids)
        x = self.pos(x)
        for b in self.blocks:
            x = b(x, pad_mask)
        x = self.norm(x)
        return self.lm_head(x)

## Part C: Decoder Training and Perplexity

Trains two decoders (RAG and baseline) and compares perplexity on the held-out test set.

In [ ]:
# Decoder train/eval helpers
def run_epoch_decoder(model, loader, optimizer=None):
    train_mode = optimizer is not None
    model.train() if train_mode else model.eval()
    total = 0.0

    for seq in loader:
        seq = seq.to(DEVICE)
        inp = seq[:, :-1]
        tgt = seq[:, 1:]

        with torch.set_grad_enabled(train_mode):
            logits = model(inp)
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1), ignore_index=PAD_ID)
            if train_mode:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

        total += loss.item()

    avg_nll = total / max(1, len(loader))
    ppl = float(np.exp(avg_nll))
    return avg_nll, ppl

def train_decoder(train_ids, val_ids, model_name='rag'):
    model = ReviewExplanationDecoder(vocab_size=len(itos)).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=2e-4)
    tr_loader = DataLoader(DecoderDataset(train_ids), batch_size=48, shuffle=True)
    va_loader = DataLoader(DecoderDataset(val_ids), batch_size=48, shuffle=False)

    best_val = float('inf')
    hist = []
    save_path = os.path.join(MODELS_DIR, f'partC_decoder_{model_name}_best.pt')

    for epoch in range(1, 5 + 1):
        tr_nll, tr_ppl = run_epoch_decoder(model, tr_loader, optimizer=opt)
        va_nll, va_ppl = run_epoch_decoder(model, va_loader, optimizer=None)
        hist.append({'epoch': epoch, 'train_nll': tr_nll, 'train_ppl': tr_ppl, 'val_nll': va_nll, 'val_ppl': va_ppl})
        print(f'[{model_name}] epoch {epoch}: train_ppl={tr_ppl:.3f}, val_ppl={va_ppl:.3f}')
        if va_nll < best_val:
            best_val = va_nll
            torch.save({'model_state_dict': model.state_dict()}, save_path)

    return model, pd.DataFrame(hist), save_path

rag_model, rag_hist, rag_path = train_decoder(train_rag_ids, val_rag_ids, model_name='rag')
base_model, base_hist, base_path = train_decoder(train_base_ids, val_base_ids, model_name='baseline')

rag_hist.to_csv(os.path.join(RESULTS_DIR, 'partC_rag_history.csv'), index=False)
base_hist.to_csv(os.path.join(RESULTS_DIR, 'partC_baseline_history.csv'), index=False)
print('Saved decoder checkpoints and histories.')

## Part C: Generation Examples, Visualizations, and RAG Ablation

Reports perplexity and prints at least five generated explanations for qualitative analysis.

In [ ]:
# Evaluation + generation
rag_model.load_state_dict(torch.load(rag_path, map_location=DEVICE)['model_state_dict'])
base_model.load_state_dict(torch.load(base_path, map_location=DEVICE)['model_state_dict'])

rag_test_loader = DataLoader(DecoderDataset(test_rag_ids), batch_size=48, shuffle=False)
base_test_loader = DataLoader(DecoderDataset(test_base_ids), batch_size=48, shuffle=False)

rag_nll, rag_ppl = run_epoch_decoder(rag_model, rag_test_loader, optimizer=None)
base_nll, base_ppl = run_epoch_decoder(base_model, base_test_loader, optimizer=None)
print(f'RAG test perplexity: {rag_ppl:.4f}')
print(f'Baseline test perplexity: {base_ppl:.4f}')

plt.figure(figsize=(8, 4))
plt.plot(rag_hist['epoch'], rag_hist['val_ppl'], marker='o', label='RAG val ppl')
plt.plot(base_hist['epoch'], base_hist['val_ppl'], marker='o', label='Baseline val ppl')
plt.title('Part C Validation Perplexity')
plt.xlabel('Epoch')
plt.ylabel('Perplexity')
plt.legend()
plt.show()

id2tok = {i: t for i, t in enumerate(itos)}

def decode_ids(ids):
    toks = []
    for i in ids:
        tok = id2tok.get(int(i), UNK_TOKEN)
        if tok in [PAD_TOKEN, BOS_TOKEN]:
            continue
        if tok == EOS_TOKEN:
            break
        toks.append(tok)
    return ' '.join(toks)

def generate_text(model, prompt_text, max_new_tokens=50):
    model.eval()
    toks = tokenize(clean_text(f'{BOS_TOKEN} {prompt_text}'))
    ids = [stoi.get(t, UNK_ID) for t in toks][:DEC_MAX_LEN-1]
    x = torch.tensor(ids, dtype=torch.long, device=DEVICE).unsqueeze(0)

    for _ in range(max_new_tokens):
        if x.size(1) >= DEC_MAX_LEN:
            break
        with torch.no_grad():
            logits = model(x)
            nxt = logits[:, -1, :].argmax(-1, keepdim=True)
        x = torch.cat([x, nxt], dim=1)
        if nxt.item() == EOS_ID:
            break

    return decode_ids(x.squeeze(0).tolist())

print('\nFive qualitative examples (RAG vs baseline):')
examples = test_gen_df.sample(5, random_state=SEED)
rows_out = []
for i, row in examples.iterrows():
    review = row['clean_text']
    ret = retrieve_similar_reviews(review, k=K_RETRIEVE)['text'].tolist()
    prompt_rag = build_prompt(review, row['pred_sentiment'], row['pred_length'], ret)
    prompt_base = build_prompt(review, row['pred_sentiment'], row['pred_length'], [])
    gen_rag = generate_text(rag_model, prompt_rag)
    gen_base = generate_text(base_model, prompt_base)
    print('\nReview:', review[:180], '...')
    print('RAG:', gen_rag[:260])
    print('BASE:', gen_base[:260])
    rows_out.append({
        'review': review,
        'rag_explanation': gen_rag,
        'baseline_explanation': gen_base
    })

ablation_df = pd.DataFrame([
    {'model': 'RAG', 'test_perplexity': rag_ppl},
    {'model': 'Baseline_NoRetrieval', 'test_perplexity': base_ppl}
])
ablation_df.to_csv(os.path.join(RESULTS_DIR, 'partC_ablation_perplexity.csv'), index=False)
pd.DataFrame(rows_out).to_csv(os.path.join(RESULTS_DIR, 'partC_generated_examples.csv'), index=False)
print('Saved Part C ablation and qualitative outputs in results/.')

## Export Metrics and Reproducibility Notes

This final cell saves key settings and confirms output files required for submission.

In [ ]:
# Save run configuration summary
config = {
    'seed': SEED,
    'max_len_encoder': MAX_LEN,
    'max_len_decoder': DEC_MAX_LEN,
    'batch_size': BATCH_SIZE,
    'retrieval_top_k': K_RETRIEVE,
    'vocab_size': len(itos),
    'paths': {
        'models': MODELS_DIR,
        'results': RESULTS_DIR
    }
}
with open(os.path.join(RESULTS_DIR, 'run_config.json'), 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2)

print('models/:', os.listdir(MODELS_DIR) if os.path.exists(MODELS_DIR) else [])
print('results/:', os.listdir(RESULTS_DIR) if os.path.exists(RESULTS_DIR) else [])
print('Notebook pipeline prepared successfully.')

## Hyperparameter Tuning Log

Use this section to record attempted configurations and validation outcomes, as required in the report.

In [ ]:
# Tuning log template and export
hp_log = pd.DataFrame([
    {'part': 'A', 'config': 'd_model=256,n_heads=8,n_layers=3,lr=2e-4,batch=64', 'val_metric': float(hist_df['val_sent_f1'].max()) if 'hist_df' in globals() else None, 'notes': 'initial run'},
    {'part': 'C-RAG', 'config': 'd_model=256,n_heads=8,n_layers=3,lr=2e-4,batch=48', 'val_metric': float(rag_hist['val_ppl'].min()) if 'rag_hist' in globals() else None, 'notes': 'initial run'},
    {'part': 'C-Base', 'config': 'd_model=256,n_heads=8,n_layers=3,lr=2e-4,batch=48', 'val_metric': float(base_hist['val_ppl'].min()) if 'base_hist' in globals() else None, 'notes': 'initial run'},
])

display(hp_log)
hp_log.to_csv(os.path.join(RESULTS_DIR, 'hyperparameter_tuning_log.csv'), index=False)
print('Saved hyperparameter_tuning_log.csv')

## Dataset Distribution Visualizations

These plots summarize class and category balance in the sampled dataset.

In [ ]:
# Visualize dataset label/category balance
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.countplot(x='category', data=raw_df, ax=axes[0])
axes[0].set_title('Category Distribution')
axes[0].tick_params(axis='x', rotation=30)

sns.countplot(x='sentiment', data=raw_df, ax=axes[1])
axes[1].set_title('Sentiment Distribution')
axes[1].set_xticklabels(['Neg', 'Neu', 'Pos'])

sns.histplot(raw_df['tok_len'], bins=40, kde=True, ax=axes[2])
axes[2].set_title('Token Length Distribution')

plt.tight_layout()
plt.show()

## Retrieval Sensitivity Study (k Values)

This section compares average top-k similarity across different retrieval depths to support selection of `k`.

In [ ]:
# Analyze retrieval quality vs k
k_values = [1, 3, 5, 7]
probe_queries = test_df['clean_text'].sample(50, random_state=SEED).tolist()
rows_k = []

for k in k_values:
    sims = []
    for q in probe_queries:
        ret = retrieve_similar_reviews(q, k=k)
        sims.extend(ret['similarity'].tolist())
    rows_k.append({'k': k, 'avg_similarity': float(np.mean(sims)), 'std_similarity': float(np.std(sims))})

k_df = pd.DataFrame(rows_k)
display(k_df)

plt.figure(figsize=(6, 4))
plt.errorbar(k_df['k'], k_df['avg_similarity'], yerr=k_df['std_similarity'], marker='o', capsize=4)
plt.title('Retrieval Similarity vs k')
plt.xlabel('k')
plt.ylabel('Cosine similarity')
plt.grid(True, alpha=0.3)
plt.show()

k_df.to_csv(os.path.join(RESULTS_DIR, 'partB_k_sensitivity.csv'), index=False)
print('Saved partB_k_sensitivity.csv')

## Part A Error Analysis Samples

Shows representative misclassified reviews for each task to support qualitative discussion in the report.

In [ ]:
# Collect misclassified examples on test split
err_rows = []
loader = DataLoader(ReviewDataset(test_df), batch_size=BATCH_SIZE, shuffle=False)
idx_global = 0
with torch.no_grad():
    for batch in loader:
        ids = batch['input_ids'].to(DEVICE)
        attn = batch['attention_mask'].to(DEVICE)
        s_y = batch['sentiment'].cpu().numpy()
        f_y = batch['length_bucket'].cpu().numpy()
        texts = batch['clean_text']
        s_logits, f_logits, _ = encoder_model(ids, attn)
        s_p = s_logits.argmax(-1).cpu().numpy()
        f_p = f_logits.argmax(-1).cpu().numpy()

        for i in range(len(texts)):
            if s_y[i] != s_p[i] or f_y[i] != f_p[i]:
                err_rows.append({
                    'text': texts[i],
                    'sent_true': int(s_y[i]), 'sent_pred': int(s_p[i]),
                    'feat_true': int(f_y[i]), 'feat_pred': int(f_p[i])
                })
            idx_global += 1

err_df = pd.DataFrame(err_rows)
print('Total error rows:', len(err_df))
display(err_df.head(10))
err_df.head(100).to_csv(os.path.join(RESULTS_DIR, 'partA_error_samples.csv'), index=False)
print('Saved partA_error_samples.csv')